In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
import time
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SPLITS = 10

# ============================================================
# 1. Load Data
# ============================================================
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_ids = test['PassengerId']
target = train['Transported'].astype(int)

train['is_train'] = 1
test['is_train'] = 0
data = pd.concat([train.drop('Transported', axis=1), test], ignore_index=True)

# ============================================================
# 2. Feature Engineering
# ============================================================

# --- Group / family structure from PassengerId ---
data['Group'] = data['PassengerId'].apply(lambda x: x.split('_')[0])
data['NumInGroup'] = data['PassengerId'].apply(lambda x: int(x.split('_')[1]))
data['GroupSize'] = data.groupby('Group')['Group'].transform('count')
data['IsAlone'] = (data['GroupSize'] == 1).astype(int)

# --- Last name / family ---
data['LastName'] = data['Name'].apply(lambda x: x.split(' ')[-1] if isinstance(x, str) else np.nan)
data['FamilySize'] = data.groupby('LastName')['LastName'].transform('count')
data.loc[data['LastName'].isna(), 'FamilySize'] = 1

# --- Cabin split ---
data[['Deck', 'CabinNum', 'Side']] = data['Cabin'].str.split('/', expand=True)
data['CabinNum'] = pd.to_numeric(data['CabinNum'], errors='coerce')

# --- Fill HomePlanet: group mode -> deck mode -> global mode ---
group_planet = data.groupby('Group')['HomePlanet'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['HomePlanet'] = data.apply(
    lambda r: group_planet[r['Group']] if pd.isna(r['HomePlanet']) and not pd.isna(group_planet.get(r['Group'], np.nan)) else r['HomePlanet'],
    axis=1
)
deck_planet_mode = data.groupby('Deck')['HomePlanet'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['HomePlanet'] = data.apply(
    lambda r: deck_planet_mode[r['Deck']] if pd.isna(r['HomePlanet']) and not pd.isna(deck_planet_mode.get(r['Deck'], np.nan)) else r['HomePlanet'],
    axis=1
)
data['HomePlanet'] = data['HomePlanet'].fillna(data['HomePlanet'].mode()[0])

# --- Fill Deck/Side using group mode -> HomePlanet mode -> global mode ---
group_deck = data.groupby('Group')['Deck'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['Deck'] = data.apply(
    lambda r: group_deck[r['Group']] if pd.isna(r['Deck']) and not pd.isna(group_deck.get(r['Group'], np.nan)) else r['Deck'],
    axis=1
)
planet_deck_mode = data.groupby('HomePlanet')['Deck'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['Deck'] = data.apply(
    lambda r: planet_deck_mode[r['HomePlanet']] if pd.isna(r['Deck']) else r['Deck'],
    axis=1
)
data['Deck'] = data['Deck'].fillna(data['Deck'].mode()[0])

group_side = data.groupby('Group')['Side'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['Side'] = data.apply(
    lambda r: group_side[r['Group']] if pd.isna(r['Side']) and not pd.isna(group_side.get(r['Group'], np.nan)) else r['Side'],
    axis=1
)
data['Side'] = data['Side'].fillna(data['Side'].mode()[0])

# --- CabinNum: fill via group median then deck median, then bin into regions ---
group_cabinnum = data.groupby('Group')['CabinNum'].transform('median')
data['CabinNum'] = data['CabinNum'].fillna(group_cabinnum)
deck_cabinnum = data.groupby('Deck')['CabinNum'].transform('median')
data['CabinNum'] = data['CabinNum'].fillna(deck_cabinnum)
data['CabinNum'] = data['CabinNum'].fillna(data['CabinNum'].median())
data['CabinRegion'] = pd.cut(data['CabinNum'], bins=[-1, 300, 600, 900, 1200, 1500, 1800, 2000],
                              labels=[0, 1, 2, 3, 4, 5, 6]).astype(float)

# --- Destination: group mode -> global mode ---
group_dest = data.groupby('Group')['Destination'].agg(lambda x: x.dropna().mode()[0] if len(x.dropna()) else np.nan)
data['Destination'] = data.apply(
    lambda r: group_dest[r['Group']] if pd.isna(r['Destination']) and not pd.isna(group_dest.get(r['Group'], np.nan)) else r['Destination'],
    axis=1
)
data['Destination'] = data['Destination'].fillna(data['Destination'].mode()[0])

# --- Spend features ---
amenities = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# CryoSleep passengers spend 0 -- fill those first
cryo_mask_true = data['CryoSleep'] == True
for col in amenities:
    data.loc[cryo_mask_true & data[col].isna(), col] = 0

# fill remaining amenity NaNs with group median, then deck median, then 0
for col in amenities:
    grp_med = data.groupby('Group')[col].transform('median')
    data[col] = data[col].fillna(grp_med)
    data[col] = data[col].fillna(data[col].median())

data['TotalSpend'] = data[amenities].sum(axis=1)
data['NoSpend'] = (data['TotalSpend'] == 0).astype(int)
for col in amenities + ['TotalSpend']:
    data[col + '_log'] = np.log1p(data[col])

# --- CryoSleep imputation using spend pattern ---
data['CryoSleep'] = data['CryoSleep'].map({True: 1, False: 0})
data.loc[(data['CryoSleep'].isna()) & (data['NoSpend'] == 1), 'CryoSleep'] = 1
data.loc[(data['CryoSleep'].isna()) & (data['NoSpend'] == 0), 'CryoSleep'] = 0
data['CryoSleep'] = data['CryoSleep'].astype(int)

# Recompute NoSpend-based zero-out: cryosleep passengers truly have 0 spend
data.loc[data['CryoSleep'] == 1, amenities] = 0
data['TotalSpend'] = data[amenities].sum(axis=1)
data['NoSpend'] = (data['TotalSpend'] == 0).astype(int)
for col in amenities + ['TotalSpend']:
    data[col + '_log'] = np.log1p(data[col])

# --- Age ---
data['Age'] = data['Age'].fillna(data.groupby('HomePlanet')['Age'].transform('median'))
data['Age'] = data['Age'].fillna(data['Age'].median())
data['AgeGroup'] = pd.cut(data['Age'], bins=[-1, 12, 18, 25, 35, 50, 100],
                           labels=[0, 1, 2, 3, 4, 5]).astype(int)
data['IsChild'] = (data['Age'] < 13).astype(int)

# --- VIP ---
data['VIP'] = data['VIP'].map({True: 1, False: 0})
data['VIP'] = data['VIP'].fillna(0).astype(int)

# --- Spend ratios / interactions ---
data['SpendPerAmenity'] = data['TotalSpend'] / (data[amenities].astype(bool).sum(axis=1) + 1)
data['LuxurySpend'] = data['Spa'] + data['VRDeck']
data['LuxurySpend_log'] = np.log1p(data['LuxurySpend'])

# --- Drop helper columns ---
data = data.drop(['Name', 'Cabin', 'PassengerId', 'Group', 'LastName'], axis=1)

cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
for col in cat_cols:
    data[col] = data[col].astype('category')

train_proc = data[data['is_train'] == 1].drop('is_train', axis=1).reset_index(drop=True)
test_proc = data[data['is_train'] == 0].drop('is_train', axis=1).reset_index(drop=True)

print(f"Final feature count: {train_proc.shape[1]}")
print("Features:", list(train_proc.columns))

# ============================================================
# 3. Model configs
# ============================================================
xgb_params = dict(
    max_depth=5, learning_rate=0.02, n_estimators=3000,
    subsample=0.75, colsample_bytree=0.6, min_child_weight=4,
    reg_alpha=0.2, reg_lambda=1.2, gamma=0.1,
    objective='binary:logistic', eval_metric='logloss',
    enable_categorical=True, tree_method='hist',
    random_state=RANDOM_STATE, early_stopping_rounds=100
)

lgb_params = dict(
    max_depth=6, learning_rate=0.02, n_estimators=3000,
    subsample=0.75, colsample_bytree=0.6, min_child_samples=20,
    reg_alpha=0.2, reg_lambda=1.2, num_leaves=31,
    objective='binary', metric='binary_logloss',
    random_state=RANDOM_STATE, verbosity=-1
)

cat_features_idx = [train_proc.columns.get_loc(c) for c in cat_cols]
cb_params = dict(
    depth=6, learning_rate=0.03, iterations=3000,
    l2_leaf_reg=4, random_state=RANDOM_STATE,
    loss_function='Logloss', eval_metric='Logloss',
    verbose=False, early_stopping_rounds=100
)

# ============================================================
# 4. Cross validation with 3-model ensemble + LR stacking
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_xgb = np.zeros(len(train_proc))
oof_lgb = np.zeros(len(train_proc))
oof_cb = np.zeros(len(train_proc))

test_xgb = np.zeros(len(test_proc))
test_lgb = np.zeros(len(test_proc))
test_cb = np.zeros(len(test_proc))

# for catboost/lgbm need object dtype categories as strings (no NaN)
train_cb = train_proc.copy()
test_cb_df = test_proc.copy()
for c in cat_cols:
    train_cb[c] = train_cb[c].astype(str)
    test_cb_df[c] = test_cb_df[c].astype(str)

fold_blend_accuracies = []

start_time = time.time()
for fold, (tr_idx, val_idx) in enumerate(skf.split(train_proc, target)):
    X_tr, X_val = train_proc.iloc[tr_idx], train_proc.iloc[val_idx]
    y_tr, y_val = target.iloc[tr_idx], target.iloc[val_idx]

    # XGBoost
    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(test_proc)[:, 1] / N_SPLITS

    # LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    test_lgb += m_lgb.predict_proba(test_proc)[:, 1] / N_SPLITS

    # CatBoost
    X_tr_cb, X_val_cb = train_cb.iloc[tr_idx], train_cb.iloc[val_idx]
    m_cb = CatBoostClassifier(**cb_params, cat_features=cat_cols)
    m_cb.fit(X_tr_cb, y_tr, eval_set=(X_val_cb, y_val), use_best_model=True)
    oof_cb[val_idx] = m_cb.predict_proba(X_val_cb)[:, 1]
    test_cb += m_cb.predict_proba(test_cb_df)[:, 1] / N_SPLITS

    fold_blend_pred = (oof_xgb[val_idx] + oof_lgb[val_idx] + oof_cb[val_idx]) / 3
    fold_blend_acc = accuracy_score(y_val, fold_blend_pred > 0.5)
    fold_blend_accuracies.append(fold_blend_acc)

    print(f"Fold {fold+1}/{N_SPLITS} done "
          f"| xgb_acc={accuracy_score(y_val, oof_xgb[val_idx]>0.5):.4f} "
          f"| lgb_acc={accuracy_score(y_val, oof_lgb[val_idx]>0.5):.4f} "
          f"| cb_acc={accuracy_score(y_val, oof_cb[val_idx]>0.5):.4f}")

training_time = time.time() - start_time

# ============================================================
# 5. Stack with Logistic Regression meta-model
# ============================================================
stack_X = np.column_stack([oof_xgb, oof_lgb, oof_cb])
stack_test_X = np.column_stack([test_xgb, test_lgb, test_cb])

meta = LogisticRegression()
# fit meta on full OOF (fine since these are already OOF/out-of-fold preds)
meta.fit(stack_X, target)
oof_stack = meta.predict_proba(stack_X)[:, 1]
test_stack = meta.predict_proba(stack_test_X)[:, 1]

# Also try simple average blend for comparison
oof_blend = (oof_xgb + oof_lgb + oof_cb) / 3
test_blend = (test_xgb + test_lgb + test_cb) / 3

print("\n" + "="*50)
print("INDIVIDUAL MODEL OOF METRICS")
print("="*50)
for name, oof in [('XGBoost', oof_xgb), ('LightGBM', oof_lgb), ('CatBoost', oof_cb)]:
    print(f"{name:10s} | Acc: {accuracy_score(target, oof>0.5):.4f} | AUC: {roc_auc_score(target, oof):.4f} | LogLoss: {log_loss(target, oof):.4f}")

print("\n" + "="*50)
print("ENSEMBLE COMPARISON")
print("="*50)
print(f"Simple Blend | Acc: {accuracy_score(target, oof_blend>0.5):.4f} | AUC: {roc_auc_score(target, oof_blend):.4f} | LogLoss: {log_loss(target, oof_blend):.4f}")
print(f"LR Stack     | Acc: {accuracy_score(target, oof_stack>0.5):.4f} | AUC: {roc_auc_score(target, oof_stack):.4f} | LogLoss: {log_loss(target, oof_stack):.4f}")

# Choose best strategy based on OOF accuracy
if accuracy_score(target, oof_stack>0.5) >= accuracy_score(target, oof_blend>0.5):
    final_oof, final_test, strategy = oof_stack, test_stack, "LR Stack"
else:
    final_oof, final_test, strategy = oof_blend, test_blend, "Simple Blend"

final_preds_oof = (final_oof > 0.5).astype(int)

print("\n" + "="*50)
print(f"FINAL METRICS FOR YOUR DOCUMENT (strategy = {strategy})")
print("="*50)
print(f"Model / algorithm: Ensemble of XGBoost + LightGBM + CatBoost ({strategy})")
print(f"Libraries / frameworks: pandas, numpy, scikit-learn, xgboost, lightgbm, catboost")
print(f"Final feature count: {train_proc.shape[1]}")
print(f"CV strategy: {N_SPLITS}-Fold Stratified Cross-Validation")
print(f"CV Accuracy (mean): {np.mean(fold_blend_accuracies):.4f}")
print(f"CV Accuracy (std): {np.std(fold_blend_accuracies):.4f}")
print(f"Validation Accuracy (Overall OOF): {accuracy_score(target, final_preds_oof):.4f}")
print(f"Precision: {precision_score(target, final_preds_oof):.4f}")
print(f"Recall: {recall_score(target, final_preds_oof):.4f}")
print(f"F1 score: {f1_score(target, final_preds_oof):.4f}")
print(f"ROC-AUC: {roc_auc_score(target, final_oof):.4f}")
print(f"Log loss: {log_loss(target, final_oof):.4f}")
print(f"Random seed set? (Y/N): Y (random_state=42)")
print(f"Training time: {training_time:.2f} seconds")

# ============================================================
# 6. Save Submission
# ============================================================
final_test_preds = (final_test > 0.5)
submission = pd.DataFrame({'PassengerId': test_ids, 'Transported': final_test_preds})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to 'submission.csv'!")